In [1]:
%pip install -r requirements.txt

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [2]:
#Load the env details
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
import os
import re
from azure.ai.formrecognizer import DocumentAnalysisClient
from azure.core.credentials import AzureKeyCredential

# Load environment variables
endpoint = os.getenv("AZUREDOCINTELLIGENCE_ENDPOINT")
api_key = os.getenv("AZUREDOCINTELLIGENCE_API_KEY")

# Create a DocumentAnalysisClient
document_analysis_client = DocumentAnalysisClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(api_key)
)

/Users/kavinkumarbaskar/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [5]:
import os
import re
import pandas as pd
import tiktoken

# Path to the directory containing PDF files
folder_path = os.path.join(os.getcwd(),'/Users/kavinkumarbaskar/Downloads/resume-data-collection/data/data/INFORMATION-TECHNOLOGY')

def get_pdf_files(folder_path):
    for path, subdirs, files in os.walk(folder_path):
        for name in files:
            if (name.endswith(".pdf")):
                yield os.path.join(path, name)

# Function to read PDF files and extract text using Azure AI Document Intelligence
def extract_text_from_pdf(pdf_path):
    with open(pdf_path, "rb") as f:
        poller = document_analysis_client.begin_analyze_document("prebuilt-layout", document=f)
    result = poller.result()
    text = ""
    for page in result.pages:
        for line in page.lines:
            text += line.content + " "
    return text

# Function to clean text and remove special characters
def clean_text(text):
    text = re.sub(r'\s+', ' ', text)  # Remove extra whitespace
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)  # Remove special characters
    return text

# Function to split text into chunks of 500 tokens
def split_text_into_token_chunks(text, max_tokens=500):
    tokenizer = tiktoken.get_encoding("cl100k_base")
    tokens = tokenizer.encode(text)
    chunks = []
    
    for i in range(0, len(tokens), max_tokens):
        chunk_tokens = tokens[i:i + max_tokens]
        chunk_text = tokenizer.decode(chunk_tokens)
        chunks.append(chunk_text)
    
    return chunks

# Count the number of PDF files in the directory
pdf_files = [f for f in get_pdf_files(folder_path)]
num_files = len(pdf_files)
print(f"Number of PDF files in the directory: {num_files}")

#Extract the file name
for pdf_file in pdf_files:
    file_name = os.path.basename(pdf_file)

# Create a DataFrame to store the chunks
data = []

for file_id, pdf_file in enumerate(pdf_files):
    print(f"Processing file {file_id + 1}/{num_files}: {file_name}")
    pdf_path = os.path.join(folder_path, pdf_file)
    text = extract_text_from_pdf(pdf_path)
    cleaned_text = clean_text(text)
    chunks = split_text_into_token_chunks(cleaned_text)
    
    print(f"Number of chunks for file {file_name}: {len(chunks)}")
    
    for chunk_id, chunk in enumerate(chunks):
        chunk_text = chunk.strip() if chunk.strip() else "NULL"
        unique_chunk_id = f"{file_id}_{chunk_id}"
        print(f"File: {file_name}, Chunk ID: {chunk_id}, Unique Chunk ID: {unique_chunk_id}, Chunk Length: {len(chunk_text)}, Chunk Text: {chunk_text[:50]}...")  # Print first 50 characters of chunk text
        data.append({
            "file_name": file_name,
            "chunk_id": chunk_id,
            "chunk_text": chunk_text,
            "unique_chunk_id": unique_chunk_id
        })

df = pd.DataFrame(data)
df.head(3)

Number of PDF files in the directory: 120
Processing file 1/120: 28897981.pdf
Number of chunks for file 28897981.pdf: 2
File: 28897981.pdf, Chunk ID: 0, Unique Chunk ID: 0_0, Chunk Length: 3056, Chunk Text: SENIOR INFORMATION TECHNOLOGY MANAGER Executive Su...
File: 28897981.pdf, Chunk ID: 1, Unique Chunk ID: 0_1, Chunk Length: 1516, Chunk Text: upgrades repairs configuration and troubleshooting...
Processing file 2/120: 28897981.pdf
Number of chunks for file 28897981.pdf: 2
File: 28897981.pdf, Chunk ID: 0, Unique Chunk ID: 1_0, Chunk Length: 3224, Chunk Text: STAFF ASSISTANT Professional Summary Highly organi...
File: 28897981.pdf, Chunk ID: 1, Unique Chunk ID: 1_1, Chunk Length: 3014, Chunk Text: or office  Obtained signatures for financial docum...
Processing file 3/120: 28897981.pdf
Number of chunks for file 28897981.pdf: 1
File: 28897981.pdf, Chunk ID: 0, Unique Chunk ID: 2_0, Chunk Length: 2286, Chunk Text: ASSISTANT FOOTBALL COACH Summary Enthusiastic reli...
Processing file 4/1

,file_name,chunk_id,chunk_text,unique_chunk_id
0,28897981.pdf,0,SENIOR INFORMATION TECHNOLOGY MANAGER Executiv...,0_0
1,28897981.pdf,1,upgrades repairs configuration and troubleshoo...,0_1
2,28897981.pdf,0,STAFF ASSISTANT Professional Summary Highly or...,1_0


In [6]:
#read the top5 rows of the dataframe
df.head(5)

,file_name,chunk_id,chunk_text,unique_chunk_id
0,28897981.pdf,0,SENIOR INFORMATION TECHNOLOGY MANAGER Executiv...,0_0
1,28897981.pdf,1,upgrades repairs configuration and troubleshoo...,0_1
2,28897981.pdf,0,STAFF ASSISTANT Professional Summary Highly or...,1_0
3,28897981.pdf,1,or office Obtained signatures for financial d...,1_1
4,28897981.pdf,0,ASSISTANT FOOTBALL COACH Summary Enthusiastic ...,2_0


In [7]:
# Add a new column 'chunk_length' to the DataFrame to view the length of each chunk
df['chunk_length'] = df['chunk_text'].apply(len)

# Display the first few rows of the DataFrame with the new column
print(df[['file_name', 'chunk_id', 'chunk_length']].head(5))

      file_name  chunk_id  chunk_length
0  28897981.pdf         0          3056
1  28897981.pdf         1          1516
2  28897981.pdf         0          3224
3  28897981.pdf         1          3014
4  28897981.pdf         0          2286


In [8]:
import tiktoken
tokenizer = tiktoken.get_encoding("cl100k_base")
sample_encode = tokenizer.encode(df.chunk_text[0]) 
decode = tokenizer.decode_tokens_bytes(sample_encode)
decode

[b'SE',
 b'NI',
 b'OR',
 b' INFORMATION',
 b' TECHNO',
 b'LOGY',
 b' MAN',
 b'AGER',
 b' Executive',
 b' Summary',
 b' Result',
 b'sf',
 b'ocused',
 b' Information',
 b' Technology',
 b' management',
 b' professional',
 b' offering',
 b' Twenty',
 b'Two',
 b' years',
 b' of',
 b' progressive',
 b' leadership',
 b' experience',
 b' Trans',
 b'forms',
 b' high',
 b'potential',
 b' staff',
 b' into',
 b' outstanding',
 b' leaders',
 b' who',
 b' demonstrate',
 b' the',
 b' creativity',
 b' and',
 b' savvy',
 b' that',
 b' is',
 b' critical',
 b' to',
 b' both',
 b' financial',
 b' and',
 b' operational',
 b' success',
 b' Accom',
 b'pl',
 b'ished',
 b' Manager',
 b' with',
 b' extensive',
 b' experience',
 b' in',
 b' front',
 b'of',
 b'house',
 b' and',
 b' back',
 b'of',
 b'house',
 b' operations',
 b' Pro',
 b'ven',
 b' ability',
 b' to',
 b' cut',
 b' costs',
 b' and',
 b' decrease',
 b' staff',
 b' turnover',
 b' Cult',
 b'iv',
 b'ates',
 b' a',
 b' company',
 b' culture',
 b' in',
 

In [9]:
len(decode)

500

In [10]:
import os
import requests
from num2words import num2words
import pandas as pd
import numpy as np
import json
from openai import AzureOpenAI

# Specify your model name
openai_embedding_model = os.getenv("AZOPENAI_EMBEDDING_MODEL_DEPLOYMENT_NAME")

# Assuming openai_url and openai_key are your environment variables
openai_url = os.getenv("AZOPENAI_ENDPOINT") + "openai/deployments/" + openai_embedding_model + "/embeddings?api-version=2023-05-15"
openai_key = os.getenv("AZOPENAI_API_KEY")

def get_embedding(text):
    """
    Get sentence embedding using the Azure OpenAI text-embedding-small model.

    Args:
        text (str): Text to embed.

    Returns:
        list: A list containing the embedding.
    """
    response = requests.post(openai_url,
        headers={"api-key": openai_key, "Content-Type": "application/json"},
        json={"input": [text]}  # Embed the extracted chunk
    )
    
    if response.status_code == 200:
        response_json = response.json()
        embedding = json.loads(str(response_json['data'][0]['embedding']))
        return embedding
    else:
        return None

# Example usage
all_filenames = []
all_chunkids = []
all_chunks = []
all_embeddings = []

# Assuming df is already defined with the required columns
for index, row in df.iterrows():
    filename = row['file_name']
    chunkid = row['unique_chunk_id']
    chunk = row['chunk_text']
    embedding = get_embedding(chunk)
    
    if embedding is not None:
        all_filenames.append(filename)
        all_chunkids.append(chunkid)
        all_chunks.append(chunk)
        all_embeddings.append(embedding)
    
    if (index + 1) % 50 == 0:  # Print progress every 50 rows
        print(f"Completed {index + 1} rows")

# Create a new DataFrame with the results
result_df = pd.DataFrame({
    'filename': all_filenames,
    'chunkid': all_chunkids,
    'chunk': all_chunks,
    'embedding': all_embeddings
})

print(result_df.head(5))  # Display the first few rows of the dataframe

Completed 50 rows
Completed 100 rows
Completed 150 rows
Completed 200 rows
Completed 250 rows
Completed 300 rows
       filename chunkid                                              chunk  \
0  28897981.pdf     0_0  SENIOR INFORMATION TECHNOLOGY MANAGER Executiv...   
1  28897981.pdf     0_1  upgrades repairs configuration and troubleshoo...   
2  28897981.pdf     1_0  STAFF ASSISTANT Professional Summary Highly or...   
3  28897981.pdf     1_1  or office  Obtained signatures for financial d...   
4  28897981.pdf     2_0  ASSISTANT FOOTBALL COACH Summary Enthusiastic ...   

                                           embedding  
0  [-0.0012362292, -0.02833059, 0.030995477, 0.01...  
1  [-0.015432058, -0.0035819835, 0.060957912, -0....  
2  [-0.02086082, -0.007986828, 0.056396514, 0.005...  
3  [-0.040895972, -0.014664852, 0.04925681, 0.023...  
4  [-0.044498064, 0.00932863, 0.0019611325, 0.024...  


In [12]:
%pip install pyodbc

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [22]:
#lets define a function to connect to SQLDB
import os
from dotenv import load_dotenv
import pyodbc
import struct
from azure.identity import DefaultAzureCredential

# Load environment variables from .env file
load_dotenv()

def get_mssql_connection():
    # Retrieve the connection string from the environment variables
    entra_connection_string = os.getenv('ENTRA_CONNECTION_STRING')
    sql_connection_string = os.getenv('SQL_CONNECTION_STRING')

    # Determine the authentication method and connect to the database
    if entra_connection_string:
        # Entra ID Service Principal Authentication
        credential = DefaultAzureCredential(exclude_interactive_browser_credential=False)    
        token = credential.get_token('https://database.windows.net/.default')
        token_bytes = token.token.encode('UTF-16LE')
        token_struct = struct.pack(f'<I{len(token_bytes)}s', len(token_bytes), token_bytes)
        SQL_COPT_SS_ACCESS_TOKEN = 1256  # This connection option is defined by Microsoft in msodbcsql.h
        conn = pyodbc.connect(entra_connection_string, attrs_before={SQL_COPT_SS_ACCESS_TOKEN: token_struct})
    
    elif sql_connection_string:
        # SQL Authentication
        conn = pyodbc.connect(sql_connection_string)
        
    else:
        raise ValueError("No valid connection string found in the environment variables.")

    return conn

In [50]:
import pyodbc
import pandas as pd

# Retrieve the connection string from the function get_mssql_connection()
conn = get_mssql_connection()

# Create a cursor object
cursor = conn.cursor()

# Enable fast_executemany
cursor.fast_executemany = True

# Loop through the DataFrame rows and insert them into the table
for index, row in result_df.iterrows():
    chunkid = row['chunkid']
    filename = row['filename']
    chunk = row['chunk']
    embedding = row['embedding']

    # CAST(CAST(? as NVARCHAR(MAX)) AS VECTOR(384))
    
    # Use placeholders for the parameters in the SQL query
    query = f"""
    INSERT INTO resumedocs (chunkid, filename, chunk, embedding)
    VALUES (?, ?, ?, CAST(CAST(? AS NVARCHAR(MAX)) as VECTOR(1536)))
    """
    # Execute the query with the parameters
    cursor.execute(query, chunkid, filename, chunk, json.dumps(embedding))

# Commit the changes
conn.commit()

# Print a success message
print("Data inserted successfully into the 'resumedocs' table.")

# Close the connection
conn.close()

Data inserted successfully into the 'resumedocs' table.


In [51]:
from prettytable import PrettyTable

import pyodbc
import pandas as pd

# Load environment variables from .env file
load_dotenv()

# Retrieve the connection string from the environment variables
conn = get_mssql_connection()

# Create a cursor object
cursor = conn.cursor()

# Use placeholders for the parameters in the SQL query
query = "SELECT TOP(10) filename, chunkid, chunk, CAST(embedding AS NVARCHAR(MAX)) as embedding FROM dbo.resumedocs ORDER BY Id"

# Execute the query with the parameters
cursor.execute(query)
queryresults = cursor.fetchall()

# Get column names from cursor.description
column_names = [column[0] for column in cursor.description]

# Create a PrettyTable object
table = PrettyTable()

# Add column names to the table
table.field_names = column_names

# Set max width for each column to truncate data
table.max_width = 20

# Add rows to the table
for row in queryresults:
    # Truncate each value to 20 characters
    truncated_row = [str(value)[:20] for value in row]
    table.add_row(truncated_row)

# Print the table
print(table)

# Commit the changes
conn.commit()
# Close the connection
conn.close()

+--------------+---------+----------------------+----------------------+
|   filename   | chunkid |        chunk         |      embedding       |
+--------------+---------+----------------------+----------------------+
| 28897981.pdf |   0_0   | SENIOR INFORMATION T | [-1.2362292e-003,-2. |
| 28897981.pdf |   0_1   | upgrades repairs con | [-1.5432058e-002,-3. |
| 28897981.pdf |   1_0   | STAFF ASSISTANT Prof | [-2.0860819e-002,-7. |
| 28897981.pdf |   1_1   | or office  Obtained  | [-4.0895972e-002,-1. |
| 28897981.pdf |   2_0   | ASSISTANT FOOTBALL C | [-4.4498064e-002,9.3 |
| 28897981.pdf |   3_0   | IT SUPPORT TECHNICIA | [-3.6832895e-002,-1. |
| 28897981.pdf |   3_1   | lete of the Year  Ac | [8.0387881e-003,-5.9 |
| 28897981.pdf |   4_0   | IT MANAGER Summary T | [-3.9760858e-002,2.7 |
| 28897981.pdf |   4_1   | 2012 HyperV installa | [-3.0031463e-002,-9. |
| 28897981.pdf |   4_2   | 2003  Supervised all | [-6.6022314e-002,-2. |
+--------------+---------+----------------------+--

In [91]:
import os
import pyodbc
import json
from dotenv import load_dotenv

def vector_search_sql(query, num_results=2):
    # Load environment variables from .env file
    load_dotenv()

    # Use the get_mssql_connection function to get the connection string details
    conn = get_mssql_connection()

    # Create a cursor object
    cursor = conn.cursor()

    # Generate the query embedding for the user's search query
    user_query_embedding = get_embedding(query)
    
    # SQL query for similarity search using the function vector_distance to calculate cosine similarity
    sql_similarity_search = f"""
    SELECT TOP(?) filename, chunkid, chunk,
           1-vector_distance('cosine', CAST(CAST( ? AS NVARCHAR(MAX)) AS VECTOR(1536)), embedding) AS similarity_score,
           vector_distance('cosine', CAST(CAST( ? AS NVARCHAR(MAX)) AS VECTOR(1536)), embedding) AS distance_score
    FROM dbo.resumedocs
    ORDER BY distance_score 
    """

    cursor.execute(sql_similarity_search, num_results, json.dumps(user_query_embedding), json.dumps(user_query_embedding))
    results = cursor.fetchall()

    # Close the database connection
    conn.close()

    return results
    
#example usage
vector_search_sql("database administrator", num_results=3)

[('28897981.pdf', '14_0', 'INFORMATION TECHNOLOGY SPECIALIST Summary An organized DBA professional with over 6years handson experience supporting Oracle databases Sql Server databases and AWS infrastructure Equipped with excellent communication and interpersonal skills a highly organized individual and team player who possesses strong analytical and problem solving skills and is who is committed in delivering quality services to customersclients Experience Information Technology Specialist 032018 to Current Company Name City State   Primary responsibilities include production support installation and configuration migration backup and recovery performance tuning cloning security upgrades and patches  Planned installed and upgraded multiple Oracle databases from 11204 to 1220  Added targets to OEM 13c and used OEM 13c to monitored databases  Created rules security profiles using OEM 13c  Performed HotCold Backup Recovery and Cloning of databases using RMAN  Planned and implemented Backu

In [94]:
import os
import time
from dotenv import load_dotenv
from openai import AzureOpenAI
from openai._exceptions import RateLimitError
from openai import OpenAI;

# Load environment variables from a .env file
load_dotenv()

# Use environment variables for the API key and endpoint
api_key = os.getenv("AZOPENAI_API_KEY")
azure_endpoint = os.getenv("AZOPENAI_ENDPOINT")
# chat_model = os.getenv("AZOPENAI_CHAT_MODEL_DEPLOYMENT_NAME")
chat_model =  "granite-3.1-8b-instruct"  # Fallback to default model name


# Create a chat completion client
# client = AzureOpenAI(
#     api_key=api_key,
#     api_version="2024-05-01-preview",
#     azure_endpoint=azure_endpoint
# )

client = OpenAI(
    base_url="http://localhost:1400/v1",
    api_key="not_needed",
)

def generate_completion(search_results, user_input, retries=3, delay=10):
    """
    Generates a chat completion with retry logic for rate limit errors.
    
    :param search_results: The search results to include in the system prompt.
    :param user_input: The user's input or query.
    :param retries: The maximum number of retries in case of rate limit errors.
    :param delay: The delay (in seconds) between retries.
    :return: The completion response as a dictionary.
    """
    system_prompt = '''
You are an intelligent & funny assistant who will exclusively answer based on the data provided in the `search_results`:
- Use the information from `search_results` to generate your top 3 responses. If the data is not a perfect match for the user's query, use your best judgment to provide helpful suggestions and include the following format:
  File: {filename}
  Chunk ID: {chunkid}
  Similarity Score: {similarity_score}
  Add a small snippet from the Relevant Text: {chunktext}
  Do not use the entire chunk
- Avoid any other external data sources.
- Add a summary about why the candidate maybe a goodfit even if exact skills and the role being hired for are not matching , at the end of the recommendations. Ensure you call out which skills match the description and which ones are missing. If the candidate doesnt have prior experience for the hiring role which we may need to pay extra attention to during the interview process.
- Add a Microsoft related interesting fact about the technology that was searched 
'''

    messages = [{"role": "system", "content": system_prompt}]
    
    # Create an empty list to store the results
    result_list = []

    # Iterate through the search results and append relevant information to the list
    for result in search_results:
        filename = result  # Assuming filename is the first column
        chunkid = result
        chunktext = result
        similarity_score = result  # Assuming similarity_score is the third column
        
        # Append the relevant information as a dictionary to the result_list
        result_list.append({
            "filename": filename,
            "chunkid": chunkid,
            "chunktext": chunktext,
            "similarity_score": similarity_score
        })

    # Add the search results to the messages
    messages.append({"role": "system", "content": f"{result_list}"})
    messages.append({"role": "user", "content": user_input})
    
    # Retry logic for rate limit errors
    for attempt in range(retries):
        try:
            response = client.chat.completions.create(
                model=chat_model,
                messages=messages,
                temperature=0
            )
            return response.dict()
        except RateLimitError as e:
            if attempt < retries - 1:
                print(f"Rate limit exceeded. Retrying in {delay} seconds... (Attempt {attempt + 1}/{retries})")
                time.sleep(delay)
            else:
                print("Rate limit exceeded. Max retries reached.")
                raise


In [95]:
# Create a loop of user input and model output to perform Q&A on the PDF's that are now chunked and stored in the SQL DB with embeddings
#
# PLEASE NOTE: An input box will be displayed for the user to enter a question/query at the top of the scree.
# The model will then provide a response based on the data stored in the SQL DB.
# Type 'end' to end the session.
#
print("*** What Role are you hiring for? And What skills are you looking for? Ask me & I can help you find a candidate :) Type 'end' to end the session.\n")

while True:
    user_input = input("User prompt: ")
    if user_input.lower() == "end":
        break

    # Print the user's question
    print(f"\nUser asked: {user_input}")

  
    # Assuming vector_search_sql and generate_completion are defined functions that work correctly
    search_results = vector_search_sql(user_input)
    completions_results = generate_completion(search_results, user_input)

    # Print the model's response
    print("\nAI's response:")
    print(completions_results['choices'][0]['message']['content'])

# The loop will continue until the user types 'end'

*** What Role are you hiring for? And What skills are you looking for? Ask me & I can help you find a candidate :) Type 'end' to end the session.


User asked:  We are hiring a Product Manager in the Microsoft Azure Database Migration team. Candidate should have good knowledge on SQL and any other databases like Oracle, PostgreSQL etc. It would be beneficial if they have had cloud experience . We seek strong handson skills in migration projects


BadRequestError: Error code: 400 - {'error': '<LM Studio error> Trying to keep the first 5062 tokens when context the overflows. However, the model is loaded with context length of only 4096 tokens, which is not enough. Try to load the model with a larger context length, or provide a shorter input. Error Data: n/a, Additional Data: n/a'}